# Tutorial: Use an AMD RX 5700 XT from PyTorch

[English](install_pytorch_mps_rx5700xt.ipynb) | [简体中文](install_pytorch_mps_rx5700xt_CN.ipynb)

## What you will accomplish (Goal)

This tutorial is for an x86_64 Hackintosh running macOS. It configures PyTorch
to use an AMD Radeon RX 5700 XT through Apple's **MPS/Metal** backend.

By the end, you will be able to:

1. create an isolated environment without changing the system Python;
2. run PyTorch in JupyterLab;
3. confirm that calculations use the `mps:0` device;
4. run a simple matrix test and observe RX 5700 XT performance.

> CUDA and ROCm are not required here. CUDA is for NVIDIA GPUs; PyTorch uses
> MPS/Metal to access this AMD GPU on macOS.

## Before you start (Setup)

### Tested computer configuration

| Item | Configuration |
|---|---|
| CPU | AMD Ryzen 9 3900X |
| GPU | AMD Radeon RX 5700 XT 8 GB |
| Architecture | x86_64 |
| macOS | 15.2 Sequoia |
| Metal | Metal 3 |
| Python | 3.11 |
| PyTorch | 2.2.2 |
| torchvision / torchaudio | 0.17.2 / 2.2.2 |
| NumPy | 1.26.4 |

### Why does the tutorial pin these versions?

PyTorch 2.2.x is the final release line with official prebuilt packages for
macOS x86_64.
This tutorial therefore uses Python 3.11 and PyTorch 2.2.2. Even if a newer
Python is already installed, do not install this environment into the system
Python.

First clone this repository and enter the tutorial directory:

```bash
git clone https://github.com/hermanzhaozzzz/My-Opencore-EFI-for-AMD3900X-5700XT-TUF-x570-Hackintosh.git
cd My-Opencore-EFI-for-AMD3900X-5700XT-TUF-x570-Hackintosh/tutorial_pytorch-via-this_AMD5700xtGPU
```

Choose either the `uv` method or the Conda method below. Do not combine them in
the same environment.

References: [PyTorch 2.2 release notes](https://pytorch.org/blog/pytorch2-2/) ·
[PyTorch MPS documentation](https://docs.pytorch.org/docs/stable/notes/mps.html)

## Installation (Steps)

### Method A: Install with uv (recommended)

Install `uv` first if it is not already available:

```bash
brew install uv
```

Then run the following commands from the tutorial directory:

```bash
uv python install 3.11
uv venv --python 3.11 .venv

uv pip install --python .venv/bin/python   "numpy==1.26.4"   "torch==2.2.2"   "torchvision==0.17.2"   "torchaudio==2.2.2"   "jupyterlab>=4,<5"   "ipykernel>=6,<8"
```

Start JupyterLab after installation:

```bash
env -u PYTORCH_ENABLE_MPS_FALLBACK .venv/bin/jupyter lab
```

Open `install_pytorch_mps_rx5700xt.ipynb` in the browser. Starting Jupyter with
`.venv/bin/jupyter` prevents it from accidentally using another Python
environment on the computer.

### Method B: Install with Conda

Use this method instead if you normally work with Anaconda or Miniconda:

```bash
conda create -n pytorch-mps-rx5700xt python=3.11 pip -y
conda activate pytorch-mps-rx5700xt

python -m pip install   "numpy==1.26.4"   "torch==2.2.2"   "torchvision==0.17.2"   "torchaudio==2.2.2"   "jupyterlab>=4,<5"   "ipykernel>=6,<8"
```

Keep the Conda environment active and start JupyterLab:

```bash
env -u PYTORCH_ENABLE_MPS_FALLBACK jupyter lab
```

Conda manages the Python environment here, while pip installs the tested
official PyTorch packages. Do not upgrade this environment directly to
PyTorch 2.3 or later.

### Step 1: Check the installation

In JupyterLab, choose **Kernel → Restart Kernel and Run All Cells**. The next
cell displays the active Python, PyTorch and MPS status.

A correct result should show:

- Python `3.11.x`;
- architecture `x86_64`;
- PyTorch `2.2.2`;
- both `MPS built` and `MPS available` set to `True`.

In [1]:
import os
import platform
import sys

import numpy as np
import torch

print(f"Python executable : {os.path.basename(sys.executable)}")
print(f"Python version    : {platform.python_version()}")
print(f"Architecture      : {platform.machine()}")
print(f"NumPy             : {np.__version__}")
print(f"PyTorch           : {torch.__version__}")
print(f"MPS built         : {torch.backends.mps.is_built()}")
print(f"MPS available     : {torch.backends.mps.is_available()}")

assert platform.machine() == "x86_64", "This tutorial was validated on x86_64 only"
assert platform.python_version_tuple()[:2] == ("3", "11"), "Use Python 3.11"
assert torch.__version__.startswith("2.2.2"), "Install PyTorch 2.2.2"
assert torch.backends.mps.is_built(), "This PyTorch build has no MPS support"
assert torch.backends.mps.is_available(), "No available MPS device was found"

fallback = os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK", "").lower()
assert fallback not in {"1", "true", "yes"}, (
    "Disable PYTORCH_ENABLE_MPS_FALLBACK and restart JupyterLab"
)

print("\nEnvironment check passed. You can continue.")

Python executable : python
Python version    : 3.11.15
Architecture      : x86_64
NumPy             : 1.26.4
PyTorch           : 2.2.2
MPS built         : True
MPS available     : True

Environment check passed. You can continue.


## Confirm that the GPU is working (Checks)

### Step 2: Run the minimal verification

The next cell creates a tensor directly on the MPS device and squares it. If
the final line contains both `device=mps:0` and `PASS`, PyTorch has sent the
calculation to the GPU.

The Ryzen 9 3900X in this test computer has no integrated GPU, so `mps:0` is
the AMD Radeon RX 5700 XT.

In [2]:
# Copy this minimal verification into another project if needed
import torch

assert torch.backends.mps.is_available(), "MPS is unavailable"

device = torch.device("mps")
x = torch.tensor([1.0, 2.0, 3.0], device=device)
y = x.square()
torch.mps.synchronize()

assert y.tolist() == [1.0, 4.0, 9.0]
print(f"PASS: {y} · device={y.device}")

PASS: tensor([1., 4., 9.], device='mps:0') · device=mps:0


### Step 3: Run a short performance check

The next cell performs ten `4096 × 4096` FP16 matrix multiplications and should
finish within a few seconds. The first two calculations warm up the GPU and are
not included in the result.

You can open iStat Menus, Activity Monitor or another hardware monitor while
the cell runs to watch GPU utilization and temperature. Results vary slightly
with background activity, GPU clocks and cooling.

In [3]:
import time

matrix_size = 4096
matrix_a = torch.randn(
    (matrix_size, matrix_size), device="mps", dtype=torch.float16
)
matrix_b = torch.randn(
    (matrix_size, matrix_size), device="mps", dtype=torch.float16
)

# Warm up twice so the initial Metal kernel compilation is not timed
torch.mps.synchronize()
for _ in range(2):
    result = matrix_a @ matrix_b
torch.mps.synchronize()

timed_iterations = 10
started = time.perf_counter()
for _ in range(timed_iterations):
    result = matrix_a @ matrix_b
torch.mps.synchronize()
elapsed = time.perf_counter() - started

operations = 2 * matrix_size ** 3 * timed_iterations
tflops = operations / elapsed / 1e12

print(f"Matrix       : {matrix_size} x {matrix_size}")
print(f"Device       : {result.device}")
print(f"Iterations   : {timed_iterations}")
print(f"Elapsed      : {elapsed:.4f} seconds")
print(f"Throughput   : {tflops:.2f} TFLOP/s")
print(f"Checksum     : {result[0, 0].item():.6g}")

assert str(result.device) == "mps:0"
assert tflops > 0

del matrix_a, matrix_b, result
torch.mps.empty_cache()

Matrix       : 4096 x 4096
Device       : mps:0
Iterations   : 10
Elapsed      : 0.2159 seconds
Throughput   : 6.36 TFLOP/s
Checksum     : -19.2656


### Results measured on this machine

The following results came from the Ryzen 9 3900X and RX 5700 XT 8 GB
Hackintosh associated with this repository. Use them as a reference; another
machine does not need to produce exactly the same numbers.

| Test | Measured result |
|---|---:|
| Minimal MPS verification | Passed on `mps:0` |
| Notebook short test | **6.36 TFLOP/s** |
| Five-minute stress-test duration | 300.74 seconds |
| Matrix multiplications in five minutes | 526 |
| Five-minute average throughput | **6.49 TFLOP/s** |
| GPU utilization during the stress test | About 95%–100% |
| Maximum observed GPU die temperature | 97°C |
| MPS / driver memory during the test | About 0.84 / 1.13 GiB |

How to read the results:

- `mps:0` is the most important result because it confirms GPU execution;
- roughly 6 TFLOP/s in this test indicates that the environment and Metal
  acceleration are working normally;
- different PyTorch operations run at very different speeds, so this matrix
  result is not the same as model-training speed;
- 97°C is high. Improve case airflow, fan settings or GPU cooling before
  training continuously for several hours.

## Use MPS in your own project (Next Steps)

### Move the model and inputs to MPS

Use this pattern in a project:

```python
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = model.to(device)
inputs = inputs.to(device)
outputs = model(inputs)
```

Model parameters and input data must be on the same device. If an error says
that the input and model are on different devices, inspect
`next(model.parameters()).device` and `inputs.device`.

### Practical notes

- The RX 5700 XT has 8 GB of VRAM. Reduce `batch_size` or input dimensions if
  an out-of-memory error occurs.
- Start with `float32`, then try `float16` only after the model works correctly.
- AMD MPS in PyTorch 2.2.2 does not support `bfloat16`.
- A small number of PyTorch operations have no MPS implementation. Setting
  `PYTORCH_ENABLE_MPS_FALLBACK=1` can temporarily run them on the CPU, but this
  reduces performance.
- When checking whether the GPU is truly in use, disable fallback and confirm
  that tensors are on `mps:0`.

The RX 5700 XT PyTorch installation and verification are now complete.